<a href="https://colab.research.google.com/github/kaushikpatriot/AI-Code/blob/main/SavingsHeuristic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Graph generator for testing
1. Generate a set of nodes
2. Generate connections between nodes such that there is a path to the Goal node
3. Generate costs for edges


In [ ]:
import random
def graphGen(nodes):
  nodeList = [i for i in range(nodes)]
  edges, coods = [], {}
  for i in range(nodes):
    x = round(random.uniform(0,15),5)
    y = round(random.uniform(0,15),5)
    coods[i] = (x,y)
  for i in range(len(nodeList)):
    for j in range(i+1, len(nodeList)):
      cost = random.randrange(20,200,10)
      edges.append((nodeList[i], nodeList[j], cost))
      #edges.append((nodeList[j], nodeList[i], cost))
  edges.sort(key=lambda x:x[0])
  adj_mat = {}
  for i in edges:
    if i[0] in adj_mat:
      adj_mat[i[0]]+=[(i[1], i[2])]
    else:
      adj_mat[i[0]]=[(i[1], i[2])]
    if i[1] in adj_mat:
      adj_mat[i[1]]+=[(i[0], i[2])]
    else:
      adj_mat[i[1]]=[(i[0], i[2])]
  #print(coods,edges)
  #print(adj_mat, edges)
  return adj_mat, edges


In [ ]:
def dataInput():
  method = input("Enter Heuristic: ").upper()
  if (method != "EUCLIDEAN" and method != "NON-EUCLIDEAN"):
    return
  nodes = int(input("Enter number of nodes: "))
  coods, edge_list, adj_list = {}, [],{}
  for i in range(nodes):
    new_cood = input("Enter co-ods: ").split(' ')
    if (len(new_cood) == 2):
      new_cood = tuple(float(i) for i in new_cood)
    else:
      return
    coods[i] = new_cood
  for i in range(nodes):
    adj_list[i] = []
    new_edges = input("Enter edges: ").split(' ')
    if (len(new_edges) == nodes):
      for j in range(len(new_edges)):
        if i != j:
          adj_list[i].append((j,float(new_edges[j])))
        if j > i:
          edge_list.append((i,j,float(new_edges[j])))
    else:
      return
  edge_list.sort(key=lambda x:x[2])
  return method, nodes, coods, edge_list, adj_list
method, nodes, coods, edge_list,adj_list = dataInput()
print(method, nodes, coods, edge_list, adj_list)

Enter Heuristic: euclidean
Enter number of nodes: 4
Enter co-ods: 1 1
Enter co-ods: 2 2
Enter co-ods: 3 3
Enter co-ods: 4 4
Enter edges: 0 10 20 30
Enter edges: 10 0 20 30
Enter edges: 10 20 0 30
Enter edges: 10 20 30 0
EUCLIDEAN 4 {0: (1.0, 1.0), 1: (2.0, 2.0), 2: (3.0, 3.0), 3: (4.0, 4.0)} [(0, 1, 10.0), (0, 2, 20.0), (1, 2, 20.0), (0, 3, 30.0), (1, 3, 30.0), (2, 3, 30.0)] {0: [(1, 10.0), (2, 20.0), (3, 30.0)], 1: [(0, 10.0), (2, 20.0), (3, 30.0)], 2: [(0, 10.0), (1, 20.0), (3, 30.0)], 3: [(0, 10.0), (1, 20.0), (2, 30.0)]}


#Overall approach
1. Toolkits for generating sample data
2. Toolkit for understanding the data given and the best algorithm that would fit
3. Wrapper function to get data and call the applicable algos

##Branch and Bound algorithm
**General Algo**
1. Inputs will contain the adjacency list
2. Get a list of edges in the ascending order of cost
3. Priority Queue will consist of (Node_name, PI, PE, cost, actual/estimated, tour)
4. Build a tour with PI and PE empty and insert the output as root node into the queue
5. Loop until final path is found (check for infinite loops)
6. Pop a node from the queue with the least cost
7. Pop an edge with the least cost from the edge list of the parent
8. Build a tour with the edge as PI and insert a node with the relevant cost
9. Build a tour with the edge as PE and insert a node with the relevant cost


**Build a Tour**

*Input:*
1. Existing Tour consisting of some linked and unlinked components
2. PIs, PEs and available edges (after removing PEs)

*Algorithm:*
1. Process PIs and create a new tour
2. Remove the PIs from the edges list
3. Iteratively pick nodes which have only one permitted edge in the list and apply them to the tour and remove PEs as consequence.
  * If there are still disconnected components then multiple possibilities exist and therefore return partial tour and estimated cost
  * If all the components are connected, then it is a unique tour, return the tour and the actual cost

*Output:*
1. Partial or a Full tour
2. Estimated or Actual cost
3. Indicator of whether the cost is estimated or actual
    
**Estimated Cost**
1. Include the PIs in both the directions
2. Include the best 1 or 2 costs as long as they are not PEs

**Actual Cost**
1. When the tour is complete i.e all nodes have found a place in the tour, then calculate the actual cost.

**Identifying PIs**
1. Explicitly included
2. Included through reasoning (How can this be arrived at?)

**Identifying PEs**
1. Explicitly excluded
2. Ones that prematurely close the circuit
3. Ones that cant be allowed as the vertices already have 2 edges from them

**Data**
1. Adjacency list (tuples of node and cost)
2. Priority queue of nodes sorted in the ascending order of costs
  * Node
  * PI list
  * PE list
  * Available edges for exploration
  * Cost
  * Actual / estimated (A/E)
  * Full tour (if actual tour available)
3. Edge list sorted in ascending order of cost








In [ ]:
adj_list = {0: [(1,10),(2,20),(3,15)], 1: [(0,10),(2,30),(3,20)],2: [(0,20),(1,30),(3,12)], 3: [(0,15),(1,20),(2,12)]}
edge_list=[]
for i in adj_list:
  for j in adj_list[i]:
    if (j[0] > i):
      edge_list.append((i,j[0],j[1]))
edge_list.sort(key=lambda x:x[2])
print (edge_list)


[(0, 1, 10), (2, 3, 12), (0, 3, 15), (0, 2, 20), (1, 3, 20), (1, 2, 30)]


In [ ]:
def cost_est():
  return 0

def cost_act(tour):
  return 0

def build_tour():
  return

def removePEs():
  return

def bnb_algo(adj_list, edge_list):
  path = False
  bnb_q = []
  bnb_q.append(['S0',[],[],edge_list,cost_est(),'E',[]])
  while path==False:
    #Get the best node
    bnb_q.sort(key=lambda x:x[3])
    node = bnb_q.pop(0)
    name, PI, PE, edges, cost, cost_type, tour = node[0],node[1],node[2],node[3],node[4],node[5],node[6]

    #Get the best edge for exploration
    edges.sort(key=lambda x: x[2])
    new_edge = edges.pop(0)

    #Treat new_edge as PI and insert node with cost
    new_tour = build_tour()
    if len(new_tour) == 1:
      pass
      actual_cost= cost_act(new_tour)
      #insert node with actual cost
    else:
      est_cost = cost_est()
      #insert node with estimated cost

    #Treat new_edge as PE and insert node with cost
    new_edges = removePEs()
    est_cost = cost_est()
    #insert node with estimated cost

    return path
    path = True

bnb_algo(adj_list, edge_list)

(0, 1, 10)


False

##Savings Heuristics
**General Algo**
1. Pick a start node from the given nodes
2. Create sub-tours with the base city and every other city and the round trip cost
3. Have an edge list not containing the base city sorted in the ascending order of cost
4. For each edge in the list calculate the savings as follows
  * Savings = Cost (base city to city 1) + cost (base city to city 2) - cost (city1 to city 2)
  * Sort this list in descending order (highest savings first)
5. For each edge in the list in the savings list above
  1. Check if atleast one of the nodes in the edge given connects to B.
    * If not, move to the next edge
    * If it does, merge the sub tours.

6. After all the edges in the list above are exhausted or when the tour has only one list, then return the tour.

**Merge sub-tours**
1. Maintaining a dictionary that has node wise count of edges set to 2 each
2. As each edge is processed, if the both the nodes have 2 edges or if one of the nodes have 1 and other has 2, process it, otherwise that edge cannot be processed.
3. If the edge is processed add to the list of edges along with the cost

In [ ]:
def savingsHeuristic(adj_list, edge_list, base_node, nodes):
  #Function to check if the tours can be merged or not
  def checkMerge():
    node1 = i[0]
    node2 = i[1]
    node1_cnt = sub_tours[i[0]][0]
    node2_cnt = sub_tours[i[1]][0]
    if (max(node1_cnt, node2_cnt) == 2) and ((min(node1_cnt, node2_cnt) >= 1)):
      return True
    elif node1_cnt==1 and node2_cnt==1:
      #Check if both node1 and node2's connected node has a connection to the base node
      node1_flag = False
      node2_flag = False
      loopFlag = True
      prev_node1, curr_node1 = -1, node1
      prev_node2, curr_node2 = -1, node2
      while loopFlag:

          conn_node1 = tour_adj[curr_node1][0]
          if conn_node1 == prev_node1:
            conn_node1 = tour_adj[curr_node1][1]
          conn_node1_cnt = sub_tours[conn_node1][0]
          #print(prev_node1, curr_node1, conn_node1)
          if conn_node1 != node2:
            if conn_node1_cnt == 1:
              node1_flag = True
              loopFlag=False
            else:
              prev_node1 = curr_node1
              curr_node1 = conn_node1
          else:
            loopFlag = False

      loopFlag = True
      while loopFlag:
          conn_node2 = tour_adj[curr_node2][0]
          if conn_node2 == prev_node2:
            conn_node2 = tour_adj[curr_node2][1]
          conn_node2_cnt = sub_tours[conn_node2][0]
          if conn_node2 != node1:
            if conn_node2_cnt == 1:
              node2_flag = True
              loopFlag = False
            else:
              prev_node2 = curr_node2
              curr_node2 = conn_node2
          else:
            loopFlag = False
      #print(node1_flag,node2_flag)
      if (node1_flag and node2_flag):
        return True
    return False

  #Initialise a dictionary with sub tours to the base_node and the cost
  sub_tours, tour_adj = {},{base_node:[]}
  for i in adj_list[base_node]:
    sub_tours[i[0]] = [2,i[1]]
    tour_adj[base_node].append(i[0])
  #print (sub_tours)

  #Calculate Savings
  new_edge_list = []
  for i in edge_list:
    if i[0] != base_node and i[1] != base_node:
      savings = sub_tours[i[0]][1] + sub_tours[i[1]][1] - i[2]
      new_edge_list.append((i[0],i[1],i[2],savings))

  new_edge_list.sort(key = lambda x: x[3], reverse = True)
  #print(new_edge_list)

  #Process the edges
  for i in new_edge_list:
    #Condition where one of the nodes have 2 edges to based node and the other has 1 or more
    #if (max(sub_tours[i[0]][0], sub_tours[i[1]][0]) == 2) and ((min(sub_tours[i[0]][0], sub_tours[i[1]][0]) >= 1)):
    if checkMerge():
      sub_tours[i[0]][0] -= 1
      if sub_tours[i[0]][0] == 0: tour_adj[base_node].remove(i[0])
      sub_tours[i[1]][0] -= 1
      if sub_tours[i[1]][0] == 0: tour_adj[base_node].remove(i[1])
      if i[0] not in tour_adj:
        tour_adj[i[0]] = [i[1]]
      else:
        tour_adj[i[0]] += [i[1]]
      if i[1] not in tour_adj:
        tour_adj[i[1]] = [i[0]]
      else:
        tour_adj[i[1]] += [i[0]]
    # print (f'Processing the edge {i}')
    # print (f'The sub tour status is  {sub_tours}')
    # print (f'The tour edge status is  {tour_adj}')
    # print (f'---------------------------------------\n')

  #Print tour
  final_tour=[base_node]
  prev_node, curr_node = -1, base_node
  for i in range(nodes):
    next_node = tour_adj[curr_node].pop()
    if next_node == prev_node:
       if len(tour_adj[curr_node]) >= 1:
        next_node = tour_adj[curr_node].pop()
       else:
        break
    # else:
    #    final_tour.append(next_node)
    final_tour.append(next_node)
    prev_node = curr_node
    curr_node = next_node
  #print(len(final_tour))
  print(*final_tour, sep = ' ')

nodes = 1000
adj_list, edge_list = graphGen(nodes)

savingsHeuristic(adj_list, edge_list, 0, nodes)


0 846 345 662 83 474 277 68 458 320 181 614 656 169 391 641 720 643 964 427 449 961 577 226 914 292 856 983 25 525 984 227 111 248 462 717 221 857 480 306 80 439 157 573 155 908 930 711 955 527 934 817 594 509 816 938 665 965 863 947 239 766 252 800 119 179 810 848 732 897 889 98 963 734 893 739 96 635 353 147 244 497 968 379 2 97 433 409 971 752 803 504 154 16 781 576 548 65 394 998 599 804 542 251 786 608 564 202 287 382 349 962 689 647 631 950 750 533 143 532 675 404 30 228 355 927 731 104 714 854 793 688 50 704 170 678 108 537 578 53 939 325 278 241 313 708 222 266 718 523 875 358 741 203 381 129 136 913 898 740 22 932 6 552 183 881 783 392 82 667 538 906 744 100 634 459 94 834 79 283 687 580 725 550 623 583 876 849 588 685 597 464 681 980 560 801 737 520 779 902 296 225 959 11 413 456 323 141 646 429 829 5 185 838 219 484 598 534 460 658 387 489 113 879 153 305 218 510 660 924 295 642 285 184 568 996 768 273 342 531 374 293 352 978 595 627 86 473 935 757 128 328 470 956 398 204 92